# Statistical Distribution

## ANOVA Test: Cart Value Across Categories

## Objective

To check whether cart values differ across product categories

## Hypothesis

### Null Hypothesis H0

Mean cart value is same across all categories

### Alternative Hypothesis H1

At least one category has different mean cart value

## Test Used

One Way ANOVA

## Significance Level

alpha = 0.05

## Decision Rule

If p-value < 0.05 reject H0
If p-value >= 0.05 fail to reject H0


In [0]:
from pyspark.sql.functions import col

carts = spark.table("ecom_catalog.ecom_dashboard.carts")
cart_items = spark.table("ecom_catalog.ecom_dashboard.cart_items")
products = spark.table("ecom_catalog.ecom_dashboard.products")

df = cart_items.alias("ci") \
    .join(carts.alias("c"), col("ci.cart_id") == col("c.cart_id")) \
    .join(products.alias("p"), col("ci.product_id") == col("p.product_id"))

pdf = df.select(
    col("p.category").alias("category"),
    col("c.cart_total").alias("cart_total")
).dropna().toPandas()


groups = [group["cart_total"].values for name, group in pdf.groupby("category")]

# ANOVA
from scipy.stats import f_oneway

f_stat, p_value = f_oneway(*groups)

print(f"F-statistic: {f_stat}")
print(f"P-value: {p_value}")

F-statistic: 3.5211281936896257
P-value: 0.043385490910680016


# Conclusion

## Test Output

F-statistic: 3.5211
P-value: 0.0434

## Decision

Since p-value is less than 0.05, we reject the null hypothesis

## Interpretation

There is a statistically significant difference in mean cart values across product categories

## Business Insight

Product category has a significant impact on cart value, indicating that some categories contribute more to overall revenue than others


###  Two-Sample T-Test: Impact of Discount on Cart Value

####  Objective:

To determine whether discounts significantly affect cart values.

---

###  Data Used:

From `carts` table:

* `cart_total`
* `cart_discounted_total`

---

###  Hypothesis:

* **Null Hypothesis (H₀):**
  There is no significant difference between mean cart total and discounted cart total.
  (μ₁ = μ₂)

* **Alternative Hypothesis (H₁):**
  There is a significant difference between mean cart total and discounted cart total.
  (μ₁ ≠ μ₂)

---

###  Test Used:

Independent Two-Sample T-Test

---

###  Assumptions:

* Data is approximately normally distributed
* Samples are independent
* Variances may be unequal (Welch’s T-Test)

---

###  Significance Level:

α = 0.05

---

###  Interpretation Rule:

* If **p-value < 0.05** → Reject H₀
* If **p-value ≥ 0.05** → Fail to reject H₀


In [0]:

df = spark.table("ecom_catalog.ecom_dashboard.carts")


pdf = df.select("cart_total", "cart_discounted_total").dropna().toPandas() # convert into pandas

from scipy.stats import ttest_ind

cart_total = pdf["cart_total"]
discounted_total = pdf["cart_discounted_total"]


t_stat, p_value = ttest_ind(cart_total, discounted_total, equal_var=False)

print(f"T-statistic: {t_stat}")
print(f"P-value: {p_value}")

T-statistic: 0.2109235899155173
P-value: 0.8336945527310686


###  Result & Conclusion:

* **T-statistic:** (fill after execution)
* **P-value:** (fill after execution)

---

###  Final Decision:

* If p-value < 0.05 → Reject Null Hypothesis
* If p-value ≥ 0.05 → Fail to Reject Null Hypothesis

---

###  Business Insight:

If we reject H₀:

> Discounts significantly impact cart value, indicating pricing strategy plays an important role in customer spending behavior.

If we fail to reject H₀:

> Discounts do not significantly change cart value, suggesting customers may spend similarly regardless of discounts.


###  Conclusion:

* **T-statistic:** 0.2109
* **P-value:** 0.8337

Since **p-value > 0.05**, we **fail to reject the null hypothesis**.

 **Insight:**
Discounts do not have a significant impact on cart value. Customers tend to spend similar amounts with or without discounts.


# Chi-Square Test: Category vs Stock Status

## Objective

To check whether product category is related to stock availability.

## Hypothesis

### Null Hypothesis H0

Product category and stock status are independent.

### Alternative Hypothesis H1

Product category and stock status are dependent.

## Test Used

Chi-Square Test of Independence

## Significance Level

alpha = 0.05

## Decision Rule

If p-value < 0.05 reject H0
If p-value >= 0.05 fail to reject H0


In [0]:

df = spark.table("ecom_catalog.ecom_dashboard.products")

pdf = df.select("category", "availability_status").dropna().toPandas()

import pandas as pd
contingency_table = pd.crosstab(pdf["category"], pdf["availability_status"])

print(contingency_table)

from scipy.stats import chi2_contingency

chi2_stat, p_value, dof, expected = chi2_contingency(contingency_table)

print(f"Chi-square statistic: {chi2_stat}")
print(f"P-value: {p_value}")

availability_status  In Stock  Low Stock
category                                
beauty                      5          0
fragrances                  4          1
furniture                   4          1
groceries                  14          1
Chi-square statistic: 1.8518518518518519
P-value: 0.6037169126946025


# Result and Conclusion

## Test Output

Chi-square statistic: fill value
P-value: fill value

## Decision

If p-value < 0.05 reject H0
If p-value >= 0.05 fail to reject H0

## Insight

If rejected
Product category and stock status are related

If not rejected
Product category and stock status are independent


# Conclusion

## Test Output

Chi-square statistic: 1.8519
P-value: 0.6037

## Decision

Since p-value >= 0.05, we fail to reject H0

## Insight

Product category and stock status are independent.
There is no significant relationship between category and availability.


# Linear Regression

In [0]:

df = spark.table("ecom_catalog.ecom_dashboard.carts")


pdf = df.select(
    "total_quantity",
    "total_products",
    "cart_discounted_total",
    "cart_total"
).dropna().toPandas()


X = pdf[["total_quantity", "total_products", "cart_discounted_total"]]
y = pdf["cart_total"]


from sklearn.linear_model import LinearRegression
model = LinearRegression()
model.fit(X, y)


y_pred = model.predict(X)


from sklearn.metrics import r2_score
r2 = r2_score(y, y_pred)

print("R-squared:", r2)
print("Coefficients:", model.coef_)

R-squared: 0.9984880048944936
Coefficients: [  48.82031942 -234.64687221    1.1109912 ]


In [0]:
import pandas as pd

new_data = pd.DataFrame({
    "total_quantity": [10],
    "total_products": [3],
    "cart_discounted_total": [5000]
})

predicted_value = model.predict(new_data)

print("Predicted Cart Value:", predicted_value[0])

Predicted Cart Value: 5698.655867078626


# Prediction

## Input

total_quantity: 10
total_products: 3
cart_discounted_total: 5000

## Predicted Output

Predicted cart value: 5698.66

## Interpretation

The model estimates cart value based on input features

This demonstrates how the model can be used to predict customer spending behavior in real scenarios


# Sentiment Analysis


In [0]:

df = spark.table("ecom_catalog.ecom_dashboard.products")


pdf = df.select("title", "rating").dropna().toPandas()


def get_sentiment(rating):
    if rating >= 4:
        return "Positive"
    elif rating >= 3:
        return "Neutral"
    else:
        return "Negative"

pdf["sentiment"] = pdf["rating"].apply(get_sentiment)


print(pdf["sentiment"].value_counts())

sentiment
Positive    13
Neutral     12
Negative     5
Name: count, dtype: int64
